# 🎯 [K08_M05] 검색 품질 평가하기
### 학생 실습 워크북 & 파이썬 실습 가이드 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K08/k08_m05_student.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 군사법령·규정 (Level 1) / K08 군 규정 검색 AI - 근거 조항 안내 규정 검색 도우미 구현 |
| **미션** | 미션 05. 검색 품질 평가하기 (소요시간: 약 2.0시간) |
| **실습 유형** | 규정 텍스트 청킹, 정보 검색(IR) 및 RAG 프롬프트 도우미 구현 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 05. 검색 품질 평가하기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **Hit@1, Hit@3, MRR로 검색 품질을 평가하고 실패 원인을 분류합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **정답 조항의 순위로 세 지표를 계산하고, 1위를 놓친 질문을 표로 정리하고 체계화하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 3개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `검색 품질 평가표, 실패 원인 분류` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 품질 평가 — 여러 지표로 검색 품질을 평가하고 실패 원인을 찾았는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(검색 품질 평가표, 실패 원인 분류)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""Hit@1, Hit@3, MRR(정답 순위의 역수 평균)로 검색 품질을 평가하고, 틀린 질문의 원인을 분류한다."""
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 자료 (실제 부대·규정·인물과 무관)
# ---------------------------------------------------------------------------
REGULATIONS = [('제1장 총칙',
  [('제1조(목적)', '이 규정은 가상 한빛대대 장병의 생활 질서와 복무 기준을 정하여 안전하고 건강한 병영 생활을 보장함을 목적으로 한다.'),
   ('제2조(적용 범위)', '이 규정은 가상 한빛대대에 소속된 모든 장병과 대대에 파견된 인원에게 적용한다.')]),
 ('제2장 일과 및 생활관',
  [('제3조(일과 시간)', '기상은 06시 30분, 점호는 21시 30분, 취침은 22시로 한다. 다만 근무자와 훈련 인원은 별도 계획에 따른다.'),
   ('제4조(생활관 청결)', '생활관 청소는 매일 아침 점호 후 20분간 실시한다. 분대장은 청소 구역을 주 단위로 지정한다.'),
   ('제5조(전열기 사용)', '생활관 내 개인 전열기 사용을 금지한다. 공용 난방기 이상은 즉시 중대 행정반에 신고한다.'),
   ('제6조(개인정비 시간)', '평일 18시부터 21시까지를 개인정비 시간으로 한다. 이 시간에는 복지시설을 이용할 수 있다.')]),
 ('제3장 휴가 및 외출',
  [('제7조(휴가 신청)', '정기 휴가는 시행 7일 전까지 휴가 신청서를 작성하여 소속 분대장을 거쳐 중대장에게 제출한다.'),
   ('제8조(휴가 조정)', '같은 기간 휴가 인원이 중대 정원의 20%를 넘으면 중대장이 추첨 또는 순번으로 조정한다.'),
   ('제9조(외출)', '외출은 주말과 공휴일 09시부터 21시까지로 하며, 외출 전 행선지와 연락처를 당직 계통에 보고한다.'),
   ('제10조(복귀 지연)', '부득이한 사유로 복귀가 늦어질 경우 복귀 예정 시각 전에 당직사관에게 연락하고 사유를 보고한다.')]),
 ('제4장 보안',
  [('제11조(정보통신 보안)', '업무용 계정 비밀번호는 분기마다 변경하고 영문·숫자·특수문자를 포함해 9자리 이상으로 한다.'),
   ('제12조(촬영 금지)', '부대 안에서 개인 휴대전화로 시설·문서·화면을 촬영하거나 외부로 전송하는 행위를 금지한다.'),
   ('제13조(외부 AI 서비스)', '부대 자료와 개인정보를 외부 AI 서비스에 입력하지 않는다. 교육 목적의 사용은 가상 자료로 한정한다.')]),
 ('제5장 안전 및 보건',
  [('제14조(동절기 안전)', '동절기 야외 활동은 방한 장구를 착용하고 2인 1조로 한다. 결빙 구간에서는 차량 서행을 원칙으로 한다.'),
   ('제15조(환자 발생)', '부상자나 환자가 발생하면 즉시 당직 계통과 의무실에 알리고, 의무병의 지시에 따른다.'),
   ('제16조(예방접종)', '부대가 계획한 예방접종은 모든 인원이 받는다. 다만 의무실이 연기를 판단한 경우에는 추가 일정에 받는다.')]),
 ('제6장 복지 및 자기개발',
  [('제17조(복지시설 이용)', '복지시설은 정해진 운영 시간에 이용하며, 시설 소독·점검 시간에는 이용을 제한한다.'),
   ('제18조(자기개발 지원)', '자격증 응시료와 온라인 강좌 수강료는 연간 한도 안에서 실비로 지원한다. 신청 시 증빙 서류를 첨부한다.'),
   ('제19조(건의 및 고충)', '장병은 소원수리함, 중대장 면담, 상담관 상담을 통해 건의와 고충을 제기할 수 있으며 불이익을 받지 않는다.')])]

# ---- 공통: 규정 조각(조 단위) 만들기, 평가 질문, 검색 함수 ----
def build_chunks():
    # [TODO : 학생 실습 5] 검색 품질 평가하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
QUESTIONS = [  # [교육용가상자료] 평가용 질문과 정답 조항
    ("휴가는 언제까지 신청해야 하나요?", 7), ("휴가 인원이 많으면 어떻게 정하나요?", 8),
    ("주말 외출 시간은 몇 시까지인가요?", 9), ("복귀가 늦어질 것 같으면 누구에게 연락하나요?", 10),
    ("비밀번호는 몇 자리 이상이어야 하나요?", 11), ("부대 안에서 휴대전화로 사진 찍어도 되나요?", 12),
    ("ChatGPT 같은 AI에 부대 자료를 넣어도 되나요?", 13), ("겨울철 야외 활동 때 지켜야 할 것은?", 14),
    ("다친 사람이 생기면 어떻게 하나요?", 15), ("예방접종을 못 받으면 어떻게 하나요?", 16),
    ("체력단련실을 못 쓰는 시간이 있나요?", 17), ("자격증 응시료를 지원받을 수 있나요?", 18),
    ("고충이 있으면 어디에 말하나요?", 19), ("점호는 몇 시에 하나요?", 3), ("생활관에서 전기난로를 써도 되나요?", 5),
]

chunks = build_chunks()
docs = [c["title"] + " " + c["text"] for c in chunks]
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)).fit(docs)
D = vec.transform(docs)


def ranks(q):
    # [TODO : 학생 실습 5] 검색 품질 평가하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
rows = []
for q, a in QUESTIONS:
    r = ranks(q)
    pos = r.index(a) + 1
    rows.append({"질문": q[:24], "정답": a, "정답 순위": pos, "1위": r[0]})
df = pd.DataFrame(rows)
hit1, hit3, mrr = (df["정답 순위"] == 1).mean(), (df["정답 순위"] <= 3).mean(), (1 / df["정답 순위"]).mean()
print(f"Hit@1 {hit1:.0%} / Hit@3 {hit3:.0%} / MRR {mrr:.2f}")
print("\n[1위를 놓친 질문]")
miss = df[df["정답 순위"] > 1]
print(miss.to_string(index=False) if len(miss) else "없음")
print("\n원인 분류: 장병 표현과 규정 용어가 다른 경우(예: 사진 찍다 ↔ 촬영, 전기난로 ↔ 전열기, AI ↔ 외부 AI 서비스)")
print("-> 미션 6에서 동의어 사전으로 질문을 확장한다.")
